In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, to_date

In [0]:
order_df = spark.read.format("delta").table("dev.tracking_gd.dimorder")
dimdate_df = spark.read.format("delta").table("dev.tracking_gd.dimdate")
invoice_df = spark.read.format("delta").table("dev.tracking_sl.invoice")


In [0]:
order = order_df.alias('order')
inv = invoice_df.alias('inv')
dd = dimdate_df.alias('dd')
tdetail_df = (
    inv.join(
        dd,
        to_date(inv.invoicedate) == dd['date'],
        "inner"
    )
    .join(
        order,
        (inv.orderno == order['orderno']),
        "inner"
    )
.select(
        col('invoiceno'),
        col('dd.datekey'),
        col('orderid').alias('order_id'),
    )
    .withColumn('ingestion_time', current_timestamp())
)


In [0]:
tdetail_df.createOrReplaceTempView("temp_diminvoice")

In [0]:
%sql
merge into dev.tracking_gd.diminvoice tgt
USING ( 
  SELECT DISTINCT * 
  FROM temp_diminvoice 
) src 
on tgt.invoiceno = src.invoiceno
when not matched then
  insert (invoiceno, datekey, order_id,ingestion_time)
  Values(src.invoiceno,src.datekey,src.order_id,src.ingestion_time)